In [1]:
import pandas as pd

In [2]:
messages = pd.read_csv("./smsspamcollection/SMSSpamCollection", sep="\t", names=["label", "message"])
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
messages.shape

(5572, 2)

In [4]:
messages["message"].loc[100]

"Please don't text me anymore. I have nothing else to say."

In [5]:
# Data cleaning and preprocessing
import re
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pagarwa1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [7]:
def preprocess_text(text):
    text = re.sub(r"[^a-zA-Z]", " ", text)  # Remove non-alphabetic characters
    text = text.lower()  # Convert to lowercase
    words = nltk.word_tokenize(text)  # Tokenize the text
    words = [ps.stem(word) for word in words if word not in set(stopwords.words("english"))]  # Stemming and stopword removal
    return " ".join(words)

In [8]:
preprocess_text(messages["message"].loc[100])

'pleas text anymor noth els say'

In [9]:
messages["message_cleaned"] = messages["message"].apply(preprocess_text)
messages.head()

,label,message,message_cleaned
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [12]:
## Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=2500, binary=True)
X = cv.fit_transform(messages["message_cleaned"]).toarray()

In [11]:
y = pd.get_dummies(messages["label"])
y = y.iloc[:, 1].values

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
from sklearn.naive_bayes import MultinomialNB

spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [15]:
y_pred = spam_detect_model.predict(X_test)

In [16]:
from sklearn.metrics import accuracy_score, classification_report

In [17]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc}")

Accuracy: 0.9811659192825112


In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.99      0.99      0.99       966
        True       0.93      0.93      0.93       149

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [19]:
## TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(max_features=2500)
X_tv = tv.fit_transform(messages["message_cleaned"]).toarray()

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X_tv, y, test_size=0.2, random_state=42)

In [21]:
spam_detect_model_tv = MultinomialNB().fit(X_train, y_train)
y_pred_tv = spam_detect_model_tv.predict(X_test)
acc_tv = accuracy_score(y_test, y_pred_tv)
print(f"TF-IDF Model Accuracy: {acc_tv}")
print(classification_report(y_test, y_pred_tv))

TF-IDF Model Accuracy: 0.9820627802690582
              precision    recall  f1-score   support

       False       0.98      1.00      0.99       966
        True       0.98      0.88      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

